In [ ]:
"""
PREFER — Pre-aspect dataset figures
===================================

Creates corpus-level diagnostic plots before aspect discovery.

Inputs:
    A raw or lightly processed Amazon Reviews'23 All_Beauty dataframe.

Expected columns:
    user_id
    parent_asin or asin
    text or review_text or sentence

Outputs:
    figures/fig1_preaspect_corpus_overview.pdf
    figures/fig1_preaspect_corpus_overview.png
    figures/preaspect_dataset_statistics.csv
"""

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# -----------------------------
# Paths
# -----------------------------
DATA_PATH = "../data/train_with_aspect_scores.csv"
FIG_DIR = "figures_paper"
os.makedirs(FIG_DIR, exist_ok=True)

# -----------------------------
# Design system
# -----------------------------
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",

    # Larger fonts
    "axes.titlesize": 19,
    "axes.titleweight": "bold",
    "axes.labelsize": 17,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 14,

    # Make axes/ticks a bit stronger
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4.5,
    "ytick.major.size": 4.5,

    # Grid
    "grid.linewidth": 0.6,
    "grid.alpha": 0.35,
    "axes.grid": True,
    "axes.axisbelow": True,

    # Export quality
    "figure.dpi": 250,
    "savefig.dpi": 250,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.08,
})

BLUE = "#378ADD"
GREEN = "#1D9E75"
PURPLE = "#7F77DD"


def save(fig, stem):
    for ext in ["pdf"]:
        fig.savefig(os.path.join(FIG_DIR, f"{stem}.{ext}"))
    plt.close(fig)
    print(f"saved {stem}.pdf/.png")


# -----------------------------
# Load data
# -----------------------------
print("Loading data...")
if DATA_PATH.endswith(".jsonl"):
    df = pd.read_json(DATA_PATH, lines=True)
elif DATA_PATH.endswith(".csv"):
    df = pd.read_csv(DATA_PATH)
else:
    raise ValueError("DATA_PATH must be .csv or .jsonl")

# -----------------------------
# Robust column detection
# -----------------------------
user_col = "user_id"

if "parent_asin" in df.columns:
    product_col = "parent_asin"
elif "asin" in df.columns:
    product_col = "asin"
else:
    raise ValueError("Could not find product column. Expected parent_asin or asin.")

if "text" in df.columns:
    text_col = "text"
elif "review_text" in df.columns:
    text_col = "review_text"
elif "sentence" in df.columns:
    text_col = "sentence"
else:
    raise ValueError("Could not find text column. Expected text, review_text, or sentence.")

assert user_col in df.columns, "Missing user_id column."

# -----------------------------
# Basic cleaning
# -----------------------------
df = df.dropna(subset=[user_col, product_col, text_col]).copy()
df[text_col] = df[text_col].astype(str)
df = df[df[text_col].str.strip().str.len() > 0]

# Optional: remove exact duplicate texts for the same product
df = df.drop_duplicates(subset=[product_col, text_col])

# Word length
df["n_words"] = df[text_col].str.split().str.len()

# Main statistics
reviews_per_product = df.groupby(product_col).size()
reviews_per_user = df.groupby(user_col).size()

# -----------------------------
# Save dataset statistics
# -----------------------------
stats = pd.DataFrame({
    "Metric": [
        "Reviews / evidence units",
        "Unique products",
        "Unique users",
        "Mean words / review",
        "Median words / review",
        "Mean reviews / product",
        "Median reviews / product",
        "Mean reviews / user",
        "Median reviews / user",
        "Max reviews / product",
        "Max reviews / user",
    ],
    "Value": [
        f"{len(df):,}",
        f"{df[product_col].nunique():,}",
        f"{df[user_col].nunique():,}",
        f"{df['n_words'].mean():.1f}",
        f"{df['n_words'].median():.0f}",
        f"{reviews_per_product.mean():.2f}",
        f"{reviews_per_product.median():.0f}",
        f"{reviews_per_user.mean():.2f}",
        f"{reviews_per_user.median():.0f}",
        f"{reviews_per_product.max():,}",
        f"{reviews_per_user.max():,}",
    ],
})

stats_path = os.path.join(FIG_DIR, "preaspect_dataset_statistics.csv")
stats.to_csv(stats_path, index=False)
print(f"saved {stats_path}")
print(stats)

# -----------------------------
# Figure 1: Pre-aspect corpus overview
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.8, 5.0))
fig.subplots_adjust(wspace=0.38, bottom=0.18, top=0.84)

# (a) Review-length distribution
ax = axes[0]
clip_words = df["n_words"].clip(upper=200)
ax.hist(clip_words, bins=50, color=BLUE, edgecolor="none", alpha=0.85)
ax.set_xlabel("Words per review")
ax.set_ylabel("Count")
ax.set_title("(a) Review-length distribution", pad=10, fontweight="bold")

# (b) Reviews per product, log-log
ax = axes[1]
max_prod = reviews_per_product.max()
bins_prod = np.logspace(0, np.log10(max_prod), 50)

ax.hist(
    reviews_per_product,
    bins=bins_prod,
    color=GREEN,
    edgecolor="none",
    alpha=0.85,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Reviews per product")
ax.set_ylabel("Products")
ax.set_title("(b) Product coverage (log-log)", pad=10, fontweight="bold")

# (c) Reviews per user, log-log
ax = axes[2]
max_user = reviews_per_user.max()
bins_user = np.logspace(0, np.log10(max_user), 50)

ax.hist(
    reviews_per_user,
    bins=bins_user,
    color=PURPLE,
    edgecolor="none",
    alpha=0.85,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Reviews per user")
ax.set_ylabel("Users")
ax.set_title("(c) User activity (log-log)", pad=10, fontweight="bold")

save(fig, "fig1_preaspect_corpus_overview")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure — Aspect concentration (d) + Aspect mass side by side
# ══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.spatial.distance import jensenshannon

# ── paths ──────────────────────────────────────────────────────────────────
DATA_PATH = "../data/train_with_aspect_scores_sentences.csv"

# ── design system ───────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.titlesize":     9,
    "axes.labelsize":     8,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "legend.fontsize":    7,
    "figure.dpi":         200,
    "savefig.dpi":        200,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "axes.linewidth":     0.5,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "grid.linewidth":     0.4,
    "grid.alpha":         0.4,
    "axes.grid":          True,
    "axes.axisbelow":     True,
})

# colour palette (BW-safe via hatch + linestyle where needed)
BLUE   = "#378ADD"
GREEN  = "#1D9E75"
PURPLE = "#7F77DD"
CORAL  = "#D85A30"
AMBER  = "#BA7517"
GRAY   = "#888780"


# ── load data ───────────────────────────────────────────────────────────────
print("Loading data …")
df = pd.read_csv(DATA_PATH)

product_col = "parent_asin"
user_col    = "user_id"
text_col    = "sentence" if "sentence" in df.columns else "review_text"
phi_cols    = [c for c in df.columns if c.startswith("asp_")]
K           = len(phi_cols)

assert product_col in df.columns, "Missing parent_asin"
assert user_col    in df.columns, "Missing user_id"
assert K > 0,                     "No asp_* columns found"

df["n_words"] = df[text_col].fillna("").astype(str).str.split().str.len()

units_per_product = df.groupby(product_col).size()
units_per_user    = df.groupby(user_col).size()
aspect_mass       = df[phi_cols].sum(axis=0)
aspect_mass       = (aspect_mass / aspect_mass.sum()).values


# ── summary statistics table ─────────────────────────────────────────────────
stats = {
    "Metric": [
        "Evidence units (reviews)",
        "Unique products",
        "Unique users",
        "Mean reviews / product",
        "Median reviews / product",
        "Mean reviews / user",
        "Median reviews / user",
        "Discovered aspects (K)",
        "Mean review length (words)",
    ],
    "Value": [
        f"{len(df):,}",
        f"{df[product_col].nunique():,}",
        f"{df[user_col].nunique():,}",
        f"{units_per_product.mean():.1f}",
        f"{units_per_product.median():.0f}",
        f"{units_per_user.mean():.2f}",
        f"{units_per_user.median():.0f}",
        str(K),
        f"{df['n_words'].mean():.1f}",
    ],
}
# pd.DataFrame(stats).to_csv(os.path.join(FIG_DIR, "dataset_statistics.csv"), index=False)
# print("  saved dataset_statistics.csv")
# print("Figure: aspect concentration + aspect mass …")
fig, axes = plt.subplots(1, 2, figsize=(7.5, 1))
fig.subplots_adjust(wspace=0.38)

# (a) Top-aspect score distribution  [was Fig 1d]
ax = axes[0]
if "top_aspect_score" in df.columns:
    scores = df["top_aspect_score"].dropna()
else:
    scores = df[phi_cols].max(axis=1)
ax.hist(scores, bins=40, color=CORAL, edgecolor="none", alpha=0.85)
ax.set_xlabel(r"Top-aspect score $\max_k\,\phi_{ik}$")
ax.set_ylabel("Count")
ax.set_title(r"(a) Aspect-concentration per review")

# (b) Aspect mass  [was Fig 2]
ax = axes[1]
order  = np.argsort(aspect_mass)[::-1]
x      = np.arange(K)
colors = [PURPLE if aspect_mass[i] >= 0.07 else BLUE if aspect_mass[i] >= 0.02 else GRAY
          for i in order]
ax.bar(x, aspect_mass[order], color=colors, edgecolor="none", width=0.75)
ax.set_xticks(x)
ax.set_xticklabels([f"$a_{{{order[i]}}}$" for i in range(K)],
                   rotation=45, ha="right", fontsize=8.0)
ax.set_ylabel("Corpus-level aspect mass")
ax.set_xlabel("Aspect (sorted by mass)")
ax.set_title(r"(b) Latent aspect distribution ($K={}$)".format(K))
# for i in range(2):
#     ax.annotate(f"{aspect_mass[order[i]]:.2f}",
#                 xy=(i, aspect_mass[order[i]]),
#                 xytext=(i, aspect_mass[order[i]] + 0.015),
#                 ha="center", fontsize=6.5, color=PURPLE)

save(fig, "fig_concentration_and_aspects")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Figure 3 — User preference heterogeneity heatmap
# ══════════════════════════════════════════════════════════════════════════════
print("Figure 3: user heterogeneity heatmap …")

user_profiles = (
    df.groupby(user_col)[phi_cols]
    .mean()
    .reset_index()
)
user_profiles = user_profiles.dropna()

phi_vals = user_profiles[phi_cols].values
phi_vals = np.clip(phi_vals, 1e-12, None)
phi_vals /= phi_vals.sum(axis=1, keepdims=True)

rng = np.random.default_rng(42)
K   = len(phi_cols)

# ── entropy of each user profile ──────────────────────────────────────────
entropies  = -np.sum(phi_vals * np.log(phi_vals) / np.log(K), axis=1)
top_asp    = phi_vals.argmax(axis=1)
top_score  = phi_vals.max(axis=1)

N_CONCENTRATED = 25   # one clearly dominant aspect
N_BROAD        = 10   # mass spread roughly equally
N_HETERO       = 25   # varied dominant aspect, moderate concentration

sampled = []

# 1. Concentrated users: low entropy, high top-score
conc_mask = entropies < np.percentile(entropies, 20)
conc_idx  = np.where(conc_mask)[0]
# stratify across aspects so the dominant dark cell moves across rows
for asp in range(K):
    asp_conc = conc_idx[top_asp[conc_idx] == asp]
    if len(asp_conc):
        sampled.append(rng.choice(asp_conc))
conc_chosen = [i for i in sampled]
# pad / trim to N_CONCENTRATED
while len(conc_chosen) < N_CONCENTRATED:
    conc_chosen.append(rng.choice(conc_idx))
conc_chosen = conc_chosen[:N_CONCENTRATED]

# 2. Broad users: high entropy (top quintile) — close to uniform
broad_mask = entropies > np.percentile(entropies, 80)
broad_idx  = np.where(broad_mask)[0]
broad_chosen = rng.choice(broad_idx, size=min(N_BROAD, len(broad_idx)), replace=False).tolist()

# 3. Heterogeneous moderate users: middle entropy, varied dominant aspect
mid_mask = (entropies >= np.percentile(entropies, 35)) & \
           (entropies <= np.percentile(entropies, 65))
mid_idx  = np.where(mid_mask)[0]
hetero_chosen = []
for asp in range(K):
    asp_mid = mid_idx[top_asp[mid_idx] == asp]
    if len(asp_mid):
        hetero_chosen.append(rng.choice(asp_mid))
while len(hetero_chosen) < N_HETERO:
    hetero_chosen.append(rng.choice(mid_idx))
hetero_chosen = hetero_chosen[:N_HETERO]

# ── assemble and sort within each group by dominant aspect ────────────────
def sort_by_dominant(indices):
    rows = phi_vals[indices]
    return [indices[i] for i in np.argsort(rows.argmax(axis=1))]

conc_chosen   = sort_by_dominant(conc_chosen)
hetero_chosen = sort_by_dominant(hetero_chosen)
# broad: sort by entropy descending (most uniform first)
broad_chosen  = sorted(broad_chosen, key=lambda i: -entropies[i])

all_indices = conc_chosen + broad_chosen + hetero_chosen
W = phi_vals[all_indices]               # shape: (N_users, K)
W_plot = W.T                            # shape: (K, N_users)  — aspects on y-axis

# ── group boundary positions ──────────────────────────────────────────────
b1 = len(conc_chosen)                   # end of concentrated block
b2 = b1 + len(broad_chosen)            # end of broad block

# ── plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(7.0, 3.6))

cmap = mcolors.LinearSegmentedColormap.from_list(
    "prefer_blue", ["#f1efe8", "#e6f1fb", "#378add", "#185fa5", "#0c447c"], N=256
)
im = ax.imshow(W_plot, aspect="auto", cmap=cmap, vmin=0, vmax=W_plot.max(),
               interpolation="nearest")

# aspect (y) ticks
ax.set_yticks(np.arange(0, K, 2))
ax.set_yticklabels([f"$a_{{{i}}}$" for i in range(0, K, 2)], fontsize=6.5)
ax.set_ylabel(r"Latent aspect $k$", fontsize=9)

# user (x) ticks — suppress individual labels, use group brackets instead
ax.set_xticks([])
ax.set_xlabel("Sampled users", fontsize=9)

# ax.set_title(
#     r"User preference vectors $\hat{w}_u \in \Delta^{K-1}$"
#     "\n"
#     r"(aspects $\times$ users, darker = higher preference weight)",
#     fontsize=8
# )
ax.grid(False)

# ── vertical dividers between groups ─────────────────────────────────────
for bnd in (b1 - 0.5, b2 - 0.5):
    ax.axvline(bnd, color="white", lw=1.5, ls="--")

# ── y-axis: show every aspect label ──────────────────────────────────────
ax.set_yticks(np.arange(K))
ax.set_yticklabels([f"$a_{{{i}}}$" for i in range(K)], fontsize=6.5)

# ── x-axis: curly-brace group labels at bottom, "Sampled users" below ────
ax.set_xticks([])
ax.set_xlabel("")          # cleared — we place "Sampled users" manually

def add_curly_bracket(ax, x0, x1, label, y_brace, y_text):
    """
    Draws a curly brace spanning [x0, x1] at y_brace (data coords)
    and places the label at y_text below it.
    Uses a Unicode curly brace scaled via an annotated text trick.
    """
    xm = (x0 + x1) / 2
    width = x1 - x0

    # left half-brace  ⌐  and right half-brace  ¬  stitched with a center peak
    # We approximate with three annotation arrows forming ⌐ peak ¬
    kw = dict(xycoords="data", textcoords="data",
              annotation_clip=False,
              arrowprops=dict(arrowstyle="-", color="#5f5e5a", lw=0.9))

    # flat left arm
    ax.annotate("", xy=(x0,  y_brace), xytext=(xm - 0.5, y_brace), **kw)
    # left diagonal to peak
    ax.annotate("", xy=(xm - 0.5, y_brace), xytext=(xm, y_brace - 0.35), **kw)
    # right diagonal from peak
    ax.annotate("", xy=(xm, y_brace - 0.35), xytext=(xm + 0.5, y_brace), **kw)
    # flat right arm
    ax.annotate("", xy=(xm + 0.5, y_brace), xytext=(x1, y_brace), **kw)

    ax.text(xm, y_text, label, ha="center", va="top",
            fontsize=9, color="#21211f", clip_on=False)

# y positions in data (aspect-index) units — below the heatmap bottom edge
Y_BRACE = K - 0.5 + 0.6     # just below the last aspect row
Y_LABEL = K - 0.5 + 1.15    # label text below the brace

add_curly_bracket(ax, -0.5,             b1 - 0.5,               "concentrated", Y_BRACE, Y_LABEL)
add_curly_bracket(ax, b1 - 0.5,        b2 - 0.5,               "diffuse",       Y_BRACE, Y_LABEL)
add_curly_bracket(ax, b2 - 0.5,        len(all_indices) - 0.5, "intermediate",  Y_BRACE, Y_LABEL)

# "Sampled users" centred beneath all three labels
ax.text((len(all_indices) - 1) / 2, K - 0.5 + 1.9,
        "Sampled users", ha="center", va="top",
        fontsize=9, color="black", clip_on=False)

# ── colorbar ─────────────────────────────────────────────────────────────
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, shrink=0.85)
cb.ax.tick_params(labelsize=8)
cb.set_label("Preference weight", fontsize=9)

fig.subplots_adjust(bottom=0.22)
save(fig, "fig3_user_heterogeneity")

In [ ]:
target_user = "AG73BVBKUOH22USSFJA5ZWL7AKXA"

target_match = np.where(user_profiles[user_col].values == target_user)[0]

if len(target_match) == 0:
    print(f"User {target_user} not found.")
else:
    target_idx = target_match[0]

    target_entropy = entropies[target_idx]
    target_top_aspect = top_asp[target_idx]
    target_top_score = top_score[target_idx]

    p20 = np.percentile(entropies, 20)
    p35 = np.percentile(entropies, 35)
    p65 = np.percentile(entropies, 65)
    p80 = np.percentile(entropies, 80)

    if target_entropy < p20:
        group = "concentrated"
    elif target_entropy > p80:
        group = "diffuse"
    elif p35 <= target_entropy <= p65:
        group = "intermediate"
    else:
        group = "not selected in the current heatmap groups"

    print("User:", target_user)
    print("Entropy:", target_entropy)
    print("Dominant aspect:", target_top_aspect)
    print("Dominant aspect weight:", target_top_score)
    print("Group:", group)
    print("\nAspect profile:")
    print(user_profiles.loc[target_idx, phi_cols])

In [ ]:



# # ══════════════════════════════════════════════════════════════════════════════
# # Figure 4 — Inter-user divergence + aspect entropy dual panel
# # ══════════════════════════════════════════════════════════════════════════════
# print("Figure 4: JS-divergence + entropy …")

# # Compute mean pairwise JS-divergence per product (subsample large products)
# print("  computing per-product JS-divergence (may take a moment) …")
# MAX_USERS_PER_PROD = 20
# product_divs = []

# prod_grouped = df.groupby(product_col)
# for pid, grp in prod_grouped:
#     phi = grp[phi_cols].dropna().values
#     if len(phi) < 2:
#         continue
#     if len(phi) > MAX_USERS_PER_PROD:
#         idx = rng.choice(len(phi), size=MAX_USERS_PER_PROD, replace=False)
#         phi = phi[idx]
#     divs = []
#     for i in range(len(phi)):
#         for j in range(i + 1, len(phi)):
#             p = phi[i] + 1e-12; p /= p.sum()
#             q = phi[j] + 1e-12; q /= q.sum()
#             divs.append(float(jensenshannon(p, q)))
#     if divs:
#         product_divs.append(np.mean(divs))

# product_divs = np.array(product_divs)

# fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), sharey=False)
# fig.subplots_adjust(wspace=0.35)

# # (a) JS-divergence histogram
# ax = axes[0]
# ax.hist(product_divs, bins=40, color=PURPLE, edgecolor="none", alpha=0.85)
# med = np.median(product_divs)
# ax.axvline(med, color=CORAL, lw=1.2, ls="--", label=f"Median = {med:.3f}")
# ax.set_xlabel(r"Mean pairwise JS-divergence $\overline{D}_{\mathrm{JS}}$")
# ax.set_ylabel("Products")
# ax.set_title(r"(a) Inter-user preference divergence per product")
# ax.legend(fontsize=7, frameon=False)

# # (b) Per-user normalised entropy (on the full set)
# ax = axes[1]
# all_profiles = user_profiles[phi_cols].values
# all_profiles = np.clip(all_profiles, 1e-12, None)
# all_profiles /= all_profiles.sum(axis=1, keepdims=True)
# all_entropies = -np.sum(all_profiles * np.log(all_profiles) / np.log(K), axis=1)
# # subsample for speed
# if len(all_entropies) > 50000:
#     idx = rng.choice(len(all_entropies), 50000, replace=False)
#     all_entropies = all_entropies[idx]

# ax.hist(all_entropies, bins=40, color=GREEN, edgecolor="none", alpha=0.85)
# ax.axvline(np.median(all_entropies), color=CORAL, lw=1.2, ls="--",
#            label=f"Median = {np.median(all_entropies):.2f}")
# ax.set_xlabel(r"Normalised entropy $H(\bar{\varphi}_u) / \log K$")
# ax.set_ylabel("Users")
# ax.set_title(r"(b) Aspect entropy across users")
# ax.legend(fontsize=7, frameon=False)

# save(fig, "fig4_preference_entropy")

# print("\nDone. All figures written to:", os.path.abspath(FIG_DIR))

In [ ]:
import pandas as pd
data=pd.read_csv("../data/train_with_aspect_scores_sentences.csv")
data.head()

In [ ]:
product=data[data["parent_asin"]=="B085BB7B1M"]
product["item_title"].iloc[0], product["user_id"].unique().shape

In [ ]:
# can you help me export all the review texts in a text file for this product?
with open("review_texts.txt", "w") as f:
    for text in product["review_text"]:
        f.write(text + "\n")

In [ ]:
data["item_title"].unique()

In [ ]:
# I want to see for each product how many rows are present. i.e. for each parent_asin count number of rows.
product_row_counts = (
    data.groupby("parent_asin")
      .size()
      .reset_index(name="num_rows")
      .sort_values("num_rows", ascending=False)
)

product_row_counts.head()


In [ ]:
data["parent_asin"].unique()

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

In [ ]:

# FIG_DIR = "figures"
# os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# DATA_PATH = "../data/train_with_aspect_scores_sentences.csv"
# df = pd.read_csv(DATA_PATH)

# product_col = "parent_asin"
# user_col = "user_id"
# text_col = "sentence" if "sentence" in df.columns else "review_text"
# phi_cols = [c for c in df.columns if c.startswith("asp_")]

# assert product_col in df.columns
# assert user_col in df.columns
# assert text_col in df.columns
# assert len(phi_cols) > 0, "No aspect score columns found."

# df["text_len"] = df[text_col].fillna("").astype(str).str.len()
# df["n_words"] = df[text_col].fillna("").astype(str).str.split().str.len()


In [ ]:
# # Figure 1: Evidence length distribution
# plt.figure(figsize=(6, 4))
# plt.hist(df["n_words"], bins=60)
# plt.xlabel("Number of words per evidence unit")
# plt.ylabel("Count")
# plt.title("Evidence-unit length distribution")
# plt.tight_layout()
# plt.savefig(f"{FIG_DIR}/evidence_length_distribution.pdf")
# plt.close()

In [ ]:
# # Figure 2: Evidence units per product
# units_per_product = df.groupby(product_col).size()

# plt.figure(figsize=(6, 4))
# plt.hist(units_per_product, bins=60)
# plt.xlabel("Evidence units per product")
# plt.ylabel("Number of products")
# plt.title("Product evidence coverage")
# plt.tight_layout()
# plt.savefig(f"{FIG_DIR}/evidence_units_per_product.pdf")
# plt.close()

In [ ]:
# # Figure 3: User activity distribution
# units_per_user = df.groupby(user_col).size()

# plt.figure(figsize=(6, 4))
# plt.hist(units_per_user, bins=60)
# plt.xlabel("Evidence units per user")
# plt.ylabel("Number of users")
# plt.title("User activity distribution")
# plt.tight_layout()
# plt.savefig(f"{FIG_DIR}/user_activity_distribution.pdf")
# plt.close()

In [ ]:
# # Figure 4: Aspect mass distribution
# aspect_mass = df[phi_cols].sum(axis=0)
# aspect_mass = aspect_mass / aspect_mass.sum()

# plt.figure(figsize=(7, 4))
# plt.bar(np.arange(len(phi_cols)), aspect_mass.values)
# plt.xlabel("Discovered aspect index")
# plt.ylabel("Corpus-level aspect mass")
# plt.title("Distribution of discovered aspects")
# plt.tight_layout()
# plt.savefig(f"{FIG_DIR}/aspect_mass_distribution.pdf")
# plt.close()

# # Dataset statistics table
# stats = {
#     "num_evidence_units": len(df),
#     "num_products": df[product_col].nunique(),
#     "num_users": df[user_col].nunique(),
#     "mean_units_per_product": units_per_product.mean(),
#     "median_units_per_product": units_per_product.median(),
#     "mean_units_per_user": units_per_user.mean(),
#     "median_units_per_user": units_per_user.median(),
#     "num_aspects": len(phi_cols),
# }

# pd.Series(stats).to_csv(f"{FIG_DIR}/dataset_statistics.csv")
# print(pd.Series(stats))

In [ ]:
"""
PREFER — Dataset Figures for Case Study Section
================================================
Generates publication-quality figures for the All_Beauty case study.
Matches the paper's serif / 200-dpi / per-method color palette design system.

Usage
-----
    python prefer_dataset_figures.py

Expects
-------
    data/train_with_aspect_scores.csv  (the processed CSV with asp_* columns)

Outputs (in figures/)
---------------------
    fig1_corpus_overview.pdf/.png      — 2×2 subplot: review count distributions
    fig2_aspect_mass.pdf/.png          — corpus-level aspect mass bar chart
    fig3_user_heterogeneity.pdf/.png   — heatmap of sampled user preference vectors
    fig4_preference_entropy.pdf/.png   — entropy + JS-divergence dual panel
    dataset_statistics.csv             — LaTeX-ready summary stats

FILE_MAP
--------
    Input  : data/train_with_aspect_scores.csv
    Output : figures/fig1_corpus_overview.{pdf,png}
             figures/fig2_aspect_mass.{pdf,png}
             figures/fig3_user_heterogeneity.{pdf,png}
             figures/fig4_preference_entropy.{pdf,png}
             figures/dataset_statistics.csv
"""

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.spatial.distance import jensenshannon

# ── paths ──────────────────────────────────────────────────────────────────
DATA_PATH = "../data/train_with_aspect_scores_sentences.csv"
FIG_DIR   = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ── design system ───────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.titlesize":     9,
    "axes.labelsize":     8,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "legend.fontsize":    7,
    "figure.dpi":         200,
    "savefig.dpi":        200,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "axes.linewidth":     0.5,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "grid.linewidth":     0.4,
    "grid.alpha":         0.4,
    "axes.grid":          True,
    "axes.axisbelow":     True,
})

# colour palette (BW-safe via hatch + linestyle where needed)
BLUE   = "#378ADD"
GREEN  = "#1D9E75"
PURPLE = "#7F77DD"
CORAL  = "#D85A30"
AMBER  = "#BA7517"
GRAY   = "#888780"

def save(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"{stem}.{ext}"))
    plt.close(fig)
    print(f"  saved {stem}.{{pdf,png}}")


# ── load data ───────────────────────────────────────────────────────────────
print("Loading data …")
df = pd.read_csv(DATA_PATH)

product_col = "parent_asin"
user_col    = "user_id"
text_col    = "sentence" if "sentence" in df.columns else "review_text"
phi_cols    = [c for c in df.columns if c.startswith("asp_")]
K           = len(phi_cols)

assert product_col in df.columns, "Missing parent_asin"
assert user_col    in df.columns, "Missing user_id"
assert K > 0,                     "No asp_* columns found"

df["n_words"] = df[text_col].fillna("").astype(str).str.split().str.len()

units_per_product = df.groupby(product_col).size()
units_per_user    = df.groupby(user_col).size()
aspect_mass       = df[phi_cols].sum(axis=0)
aspect_mass       = (aspect_mass / aspect_mass.sum()).values


# ── summary statistics table ─────────────────────────────────────────────────
stats = {
    "Metric": [
        "Evidence units (reviews)",
        "Unique products",
        "Unique users",
        "Mean reviews / product",
        "Median reviews / product",
        "Mean reviews / user",
        "Median reviews / user",
        "Discovered aspects (K)",
        "Mean review length (words)",
    ],
    "Value": [
        f"{len(df):,}",
        f"{df[product_col].nunique():,}",
        f"{df[user_col].nunique():,}",
        f"{units_per_product.mean():.1f}",
        f"{units_per_product.median():.0f}",
        f"{units_per_user.mean():.2f}",
        f"{units_per_user.median():.0f}",
        str(K),
        f"{df['n_words'].mean():.1f}",
    ],
}
pd.DataFrame(stats).to_csv(os.path.join(FIG_DIR, "dataset_statistics.csv"), index=False)
print("  saved dataset_statistics.csv")


# ══════════════════════════════════════════════════════════════════════════════
# Figure 1 — Corpus overview (2 × 2)
# ══════════════════════════════════════════════════════════════════════════════
print("Figure 1: corpus overview …")
fig, axes = plt.subplots(2, 2, figsize=(6.5, 4.5))
fig.subplots_adjust(hspace=0.42, wspace=0.35)

# (a) review length distribution
ax = axes[0, 0]
clip = df["n_words"].clip(upper=200)
ax.hist(clip, bins=50, color=BLUE, edgecolor="none", alpha=0.85)
ax.set_xlabel("Words per review")
ax.set_ylabel("Count")
ax.set_title(r"(a) Review-length distribution")

# (b) reviews per product (log-log)
ax = axes[0, 1]
ax.hist(units_per_product, bins=np.logspace(0, np.log10(units_per_product.max()), 50),
        color=GREEN, edgecolor="none", alpha=0.85)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Reviews per product")
ax.set_ylabel("Products")
ax.set_title(r"(b) Product coverage (log–log)")

# (c) reviews per user (log)
ax = axes[1, 0]
ax.hist(units_per_user, bins=np.logspace(0, np.log10(units_per_user.max()), 50),
        color=PURPLE, edgecolor="none", alpha=0.85)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Reviews per user")
ax.set_ylabel("Users")
ax.set_title(r"(c) User-activity distribution (log–log)")

# (d) top-aspect score distribution
ax = axes[1, 1]
if "top_aspect_score" in df.columns:
    ax.hist(df["top_aspect_score"].dropna(), bins=40, color=CORAL, edgecolor="none", alpha=0.85)
    ax.set_xlabel(r"Top-aspect score $\max_k\,\varphi_{ik}$")
else:
    top_scores = df[phi_cols].max(axis=1)
    ax.hist(top_scores, bins=40, color=CORAL, edgecolor="none", alpha=0.85)
    ax.set_xlabel(r"Top-aspect score $\max_k\,\varphi_{ik}$")
ax.set_ylabel("Count")
ax.set_title(r"(d) Aspect-concentration per review")

save(fig, "fig1_corpus_overview")


# ══════════════════════════════════════════════════════════════════════════════
# Figure 2 — Aspect mass distribution
# ══════════════════════════════════════════════════════════════════════════════
print("Figure 2: aspect mass …")
fig, ax = plt.subplots(figsize=(6.5, 2.5))

order  = np.argsort(aspect_mass)[::-1]
x      = np.arange(K)
colors = [PURPLE if aspect_mass[i] >= 0.07 else BLUE if aspect_mass[i] >= 0.02 else GRAY
          for i in order]

bars = ax.bar(x, aspect_mass[order], color=colors, edgecolor="none", width=0.75)

ax.set_xticks(x)
ax.set_xticklabels([f"$a_{{{order[i]}}}$" for i in range(K)], rotation=45, ha="right", fontsize=6.5)
ax.set_ylabel("Corpus-level aspect mass")
ax.set_xlabel("Discovered aspect (sorted by mass)")
ax.set_title(r"Corpus-level distribution of discovered latent aspects ($K={}$)".format(K))

# annotate top 2
for i in range(2):
    ax.annotate(f"{aspect_mass[order[i]]:.2f}",
                xy=(i, aspect_mass[order[i]]),
                xytext=(i, aspect_mass[order[i]] + 0.015),
                ha="center", fontsize=6.5, color=PURPLE)

save(fig, "fig2_aspect_mass")


# ══════════════════════════════════════════════════════════════════════════════
# Figure 3 — User preference heterogeneity heatmap
# ══════════════════════════════════════════════════════════════════════════════
print("Figure 3: user heterogeneity heatmap …")

# Build per-user aspect profiles: mean φ across their reviews
user_profiles = (
    df.groupby(user_col)[phi_cols]
    .mean()
    .reset_index()
)
user_profiles = user_profiles.dropna()

# Sample N representative users: stratified by top aspect
N_SAMPLE = 60
rng = np.random.default_rng(42)
top_asp = user_profiles[phi_cols].values.argmax(axis=1)
sampled_rows = []
for asp in range(K):
    idx = np.where(top_asp == asp)[0]
    if len(idx) == 0:
        continue
    n_pick = max(1, int(N_SAMPLE * (idx.shape[0] / len(user_profiles))))
    chosen = rng.choice(idx, size=min(n_pick, len(idx)), replace=False)
    sampled_rows.extend(chosen.tolist())

sampled_rows = sampled_rows[:N_SAMPLE]
W = user_profiles.iloc[sampled_rows][phi_cols].values  # (N_SAMPLE, K)

# Sort users by their dominant aspect for visual coherence
dominant = W.argmax(axis=1)
sort_idx = np.argsort(dominant)
W = W[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.8),
                         gridspec_kw={"width_ratios": [3, 1], "wspace": 0.08})

# Left: heatmap
ax = axes[0]
cmap = mcolors.LinearSegmentedColormap.from_list(
    "prefer_blue", ["#f1efe8", "#e6f1fb", "#378add", "#185fa5", "#0c447c"], N=256
)
im = ax.imshow(W, aspect="auto", cmap=cmap, vmin=0, interpolation="nearest")
ax.set_xlabel(r"Discovered aspect index $k$")
ax.set_ylabel(r"Sampled user (sorted by dominant aspect)")
ax.set_xticks(np.arange(0, K, 4))
ax.set_xticklabels([f"$a_{{{i}}}$" for i in range(0, K, 4)], fontsize=6.5)
ax.set_yticks([])
ax.set_title(r"(a) Sampled user preference vectors $\hat{w}_u \in \Delta^{K-1}$")
ax.grid(False)

cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.ax.tick_params(labelsize=6.5)
cb.set_label("Preference weight", fontsize=7)

# Right: entropy histogram (sideways)
ax2 = axes[1]
entropies = -np.sum(W * np.log(W + 1e-12) / np.log(K), axis=1)  # normalised [0,1]
ax2.hist(entropies, bins=15, orientation="horizontal",
         color=PURPLE, edgecolor="none", alpha=0.85)
ax2.set_xlabel("Users")
ax2.set_ylabel(r"Normalised entropy $H(\hat{w}_u) / \log K$", labelpad=3)
ax2.set_title(r"(b) Pref. entropy")
ax2.yaxis.set_label_position("right")
ax2.yaxis.tick_right()
ax2.set_ylim(0, 1)

save(fig, "fig3_user_heterogeneity")


# ══════════════════════════════════════════════════════════════════════════════
# Figure 4 — Inter-user divergence + aspect entropy dual panel
# ══════════════════════════════════════════════════════════════════════════════
print("Figure 4: JS-divergence + entropy …")

# Compute mean pairwise JS-divergence per product (subsample large products)
print("  computing per-product JS-divergence (may take a moment) …")
MAX_USERS_PER_PROD = 20
product_divs = []

prod_grouped = df.groupby(product_col)
for pid, grp in prod_grouped:
    phi = grp[phi_cols].dropna().values
    if len(phi) < 2:
        continue
    if len(phi) > MAX_USERS_PER_PROD:
        idx = rng.choice(len(phi), size=MAX_USERS_PER_PROD, replace=False)
        phi = phi[idx]
    divs = []
    for i in range(len(phi)):
        for j in range(i + 1, len(phi)):
            p = phi[i] + 1e-12; p /= p.sum()
            q = phi[j] + 1e-12; q /= q.sum()
            divs.append(float(jensenshannon(p, q)))
    if divs:
        product_divs.append(np.mean(divs))

product_divs = np.array(product_divs)

fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), sharey=False)
fig.subplots_adjust(wspace=0.35)

# (a) JS-divergence histogram
ax = axes[0]
ax.hist(product_divs, bins=40, color=PURPLE, edgecolor="none", alpha=0.85)
med = np.median(product_divs)
ax.axvline(med, color=CORAL, lw=1.2, ls="--", label=f"Median = {med:.3f}")
ax.set_xlabel(r"Mean pairwise JS-divergence $\overline{D}_{\mathrm{JS}}$")
ax.set_ylabel("Products")
ax.set_title(r"(a) Inter-user preference divergence per product")
ax.legend(fontsize=7, frameon=False)

# (b) Per-user normalised entropy (on the full set)
ax = axes[1]
all_profiles = user_profiles[phi_cols].values
all_profiles = np.clip(all_profiles, 1e-12, None)
all_profiles /= all_profiles.sum(axis=1, keepdims=True)
all_entropies = -np.sum(all_profiles * np.log(all_profiles) / np.log(K), axis=1)
# subsample for speed
if len(all_entropies) > 50000:
    idx = rng.choice(len(all_entropies), 50000, replace=False)
    all_entropies = all_entropies[idx]

ax.hist(all_entropies, bins=40, color=GREEN, edgecolor="none", alpha=0.85)
ax.axvline(np.median(all_entropies), color=CORAL, lw=1.2, ls="--",
           label=f"Median = {np.median(all_entropies):.2f}")
ax.set_xlabel(r"Normalised entropy $H(\bar{\varphi}_u) / \log K$")
ax.set_ylabel("Users")
ax.set_title(r"(b) Aspect entropy across users")
ax.legend(fontsize=7, frameon=False)

save(fig, "fig4_preference_entropy")

print("\nDone. All figures written to:", os.path.abspath(FIG_DIR))